In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv("messy_ecommerce_sales_data.csv")
df.head()

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


In [3]:
df.dtypes

ID                  int64
 Customer_Name        str
Order_ID              str
Order_Date            str
 Product              str
Category              str
Quantity              str
Price                 str
Payment_Method        str
Status                str
Total             float64
dtype: object

Eres un motor de limpieza de datos. Procesa el CSV que te entregaré siguiendo estas reglas:

**1. DIAGNÓSTICO**
- Identifica columnas, tipos de datos, delimitador.
- Lista problemas visibles (nulos, formatos mixtos, valores no numéricos, fechas inconsistentes).

**2. LIMPIEZA**
- Elimina filas con ID, Order_ID o Customer_Name nulos.
- Quantity y Price: convierte a número. Texto no convertible → null. Cantidades negativas → corrige o elimina.
- Total: si no coincide con Quantity × Price, recalcula.
- Elimina duplicados por Order_ID (conserva la fila más completa).
- Elimina filas con fechas inválidas o fuera de 2023-2026.

**3. NORMALIZACIÓN**
- Fechas a YYYY-MM-DD.
- Texto a minúsculas (IDs exceptuados).
- Unifica categorías (electronics, home, sports, books, clothing).
- Price y Total con 2 decimales.

**4. INFORME FINAL**
- Resumen: filas eliminadas y modificadas.
- Anomalías lógicas detectadas.
- Tabla con datos limpios.



In [10]:


def limpiar_dataset(ruta_csv):
    # =========================
    # 1. CARGA
    # =========================
    df = pd.read_csv(ruta_csv)
    df.columns = df.columns.str.strip()

    filas_iniciales = len(df)

    # =========================
    # 2. LIMPIEZA
    # =========================

    # Eliminar nulos críticos
    df = df.dropna(subset=['ID', 'Order_ID', 'Customer_Name'])

    # Convertir a numérico
    df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

    # 🔥 CLAVE: eliminar filas no convertibles
    df = df.dropna(subset=['Quantity', 'Price'])

    # Eliminar cantidades negativas
    df = df[df['Quantity'] >= 0]

    # Recalcular Total SIEMPRE
    df['Total'] = df['Quantity'] * df['Price']

    # Eliminar duplicados (mantener fila más completa)
    df['non_nulls'] = df.notna().sum(axis=1)
    df = df.sort_values(by='non_nulls', ascending=False)
    df = df.drop_duplicates(subset='Order_ID', keep='first')
    df = df.drop(columns='non_nulls')

    # Procesar fechas
    df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')

    # Eliminar fechas inválidas o fuera de rango
    df = df[
        (df['Order_Date'].dt.year >= 2023) &
        (df['Order_Date'].dt.year <= 2026)
    ]
    df = df.dropna(subset=['Order_Date'])

    # =========================
    # 3. NORMALIZACIÓN
    # =========================

    # Formato fecha
    df['Order_Date'] = df['Order_Date'].dt.strftime('%Y-%m-%d')

    # Minúsculas (excepto IDs)
    for col in df.select_dtypes(include='object').columns:
        if col not in ['ID', 'Order_ID']:
            df[col] = df[col].str.strip().str.lower()

    # Normalizar categorías (con mapeo inteligente)
    categorias_map = {
        'electronics': 'electronics',
        'electronic': 'electronics',
        'home': 'home',
        'sports': 'sports',
        'books': 'books',
        'book': 'books',
        'clothing': 'clothing',
        'clothes': 'clothing'
    }

    df['Category'] = df['Category'].map(categorias_map)

    # Formato decimal
    df['Price'] = df['Price'].round(2)
    df['Total'] = df['Total'].round(2)

    # =========================
    # 4. INFORME
    # =========================

    filas_finales = len(df)

    print("=== INFORME ===")
    print(f"Filas iniciales: {filas_iniciales}")
    print(f"Filas finales: {filas_finales}")
    print(f"Filas eliminadas: {filas_iniciales - filas_finales}")

    # Chequeos de calidad
    print("\n=== VALIDACIONES ===")
    print("NaN en Price:", df['Price'].isna().sum())
    print("NaN en Quantity:", df['Quantity'].isna().sum())
    print("Valores no válidos en Category:", df['Category'].isna().sum())

    return df


# =========================
# USO
# =========================

df_limpio = limpiar_dataset("messy_ecommerce_sales_data.csv")

# Guardar resultado
df_limpio.to_csv("clean_ecommerce_data.csv", index=False)

print("\nArchivo limpio generado: clean_ecommerce_data.csv")

=== INFORME ===
Filas iniciales: 103
Filas finales: 82
Filas eliminadas: 21

=== VALIDACIONES ===
NaN en Price: 0
NaN en Quantity: 0
Valores no válidos en Category: 7

Archivo limpio generado: clean_ecommerce_data.csv


C:\Users\PC\AppData\Local\Temp\ipykernel_16600\840489411.py:54: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:
